In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from steps.ingest import ingest
from steps.transform import transform_fn
from steps.train import estimator_fn

TABLE_NAME = "mlops_test.default.rc_test_data" 
TARGET_COL = "target"
MODEL_NAME = "mlops_test.catlog_models.randomforestregressor"

# --- pipeline ---
df = ingest(spark, TABLE_NAME)

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run():
    model = estimator_fn()
    model.fit(transform_fn(X_train), y_train)

    preds = model.predict(transform_fn(X_test))
    rmse = mean_squared_error(y_test, preds)
    print(f"RMSE: {rmse}")

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "model", input_example=X, registered_model_name=MODEL_NAME)

print("Done — check the Experiments and Models tabs in the sidebar.")

In [0]:
import argparse
import sys
import logging
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayUsageTrackingConfig,
)
from databricks.sdk.service.catalog import MonitorInferenceLog, MonitorInferenceLogProblemType
from databricks.sdk.errors import NotFound
from pyspark.sql import SparkSession
  
w = WorkspaceClient()
parsed_view = "mlops_test.capstone.capstone_payload_parsed"

existing_monitor = w.quality_monitors.get(table_name=parsed_view)
print(f"FOUND: {existing_monitor}")